In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyscf import ao2mo, gto
import braketlab as bk

# some required functions
# author: Audun Skau Hansen, 2022

def lowdin_s12_matrix(S):
    """
    Löwdin orthogonalizing matrix
    S = AO overlap matrix
    returns S^{-.5}
    """
    
    eps, U = np.linalg.eigh(S)
    return np.dot(U, np.dot(np.diag(np.reciprocal(np.sqrt(eps))), U.T))

def normalize(C):
    """
    Square-normalize columns in matrix C
    returns normalized C
    """
    norms = np.sqrt(np.diag(np.dot(C.T, np.dot(S, C))))
    
    for i in np.arange(C.shape[0]):
        C[:,i]/=norms[i]
    return C

def L2norm(M):
    """
    Compute and return L2-norm of matrix M
    """
    
    return np.sqrt(np.sum(M**2))

def inner(C_1,S,C_2):
    """
    Compute inner product
    
    ( phi_1  , phi_2 ) 
    
    where
    
    phi_i = \sum_{\alpha} Ci_{\alpha p} \chi_alpha
    
    S_{\alpha, \beta} = (\chi_{\alpha}, \chi_{\beta})
    
    """
    
    return np.dot(C_1.T, np.dot(S, C_2))

def density(c, nocc):
    """
    compute density matrix
    
    C = coefficient
    nocc = number of occupied orbitals (in RHF: number of electrons * .5 )
    """
    
    c_occ = c[:, :nocc]
    return np.dot(c_occ, c_occ.T)

def build_fock_matrix(h, eri, d):
    # construct the Fock matrix
    return h +  2*np.einsum("kl, ijkl->ij", d, eri) - np.einsum("kl, ikjl->ij", d, eri)

def extract_pyscf_basis(mo):
    """
    Get a braketlab basis from a pyscf mol object
    """
    contracted_aos = []
    for contr in range(mo.nbas):
        l = mo.bas_angular(contr)
        contr_coeff = mo.bas_ctr_coeff(contr)
        exponent = mo.bas_exp(contr)
        #position = mo.
        
        if l == 1:
            
            for m in [1, -1,0]:
                #for b in range(len(contr_coeff[a]))
                #a = 0
                for b in range(len(contr_coeff[0])):
                    a = 0
                    #print(l,m,"    ", exponent[a], contr_coeff[a][b], np.array(mo.bas_coord(contr)))
                    chi_lm = contr_coeff[a][b]*bk.basisbank.get_gto(exponent[0], l, m, position = np.array(mo.bas_coord(contr)))
                    for  a in range(1, len(exponent)):

                        #print(l,m,"    ", exponent[a], "     ", contr_coeff[a][0])
                        chi_lm += contr_coeff[a][b]*bk.basisbank.get_gto(exponent[a], l, m,  position = np.array(mo.bas_coord(contr)))
                    contracted_aos.append(chi_lm)
        else:
            for m in range(-l, l+1):
                for b in range(len(contr_coeff[0])):
                    a = 0
                    #print(l,m,"    ", exponent[a], contr_coeff[a][0], mo.bas_coord(contr))
                    chi_lm = contr_coeff[a][b]*bk.basisbank.get_gto(exponent[0],l,m, position = np.array(mo.bas_coord(contr)))
                    for  a in range(1, len(exponent)):
                        #print(l,m,"    ", exponent[a], "     ", contr_coeff[a][0])
                        chi_lm += contr_coeff[a][b]*bk.basisbank.get_gto(exponent[a], l,m, position = np.array(mo.bas_coord(contr)))

                    contracted_aos.append(chi_lm)
    return contracted_aos


def advance(h,eri,d, n_occ, s_12):
    """
    One iteration in the Hartree-Fock optimization
    
    h = core hamiltonian
    eri = electron repulsion integrals 
    d = density matrix
    n_occ = number of occupied orbitals
    s_12 = Löwdin orthogonalization matrix
    
    Returns the density and coefficients
    """
    
    # build Fock matrix
    f = build_fock_matrix(h, eri, d)
    
    # Similarity transform of the Fock matrix
    f_ = s_12.T.dot(f.dot(s_12)) 

    # Eigenvalues and eigenvectors of f'
    e_, c_ = np.linalg.eigh(f_) # compute orbital 

    # sort MOs in order of increasing eigenvalues
    c_ = c_[:,np.argsort(e_)] 

    # new set of orbitals and denisty matrix
    c0 = s_12.dot(c_)
    
    # compute density matrix
    d = density(c0, n_occ)
    
    return d, c0

In [ ]:
## Orthonormality

# set up a H2-molecule (just for integrals and some properties)
mol = gto.M(atom='H 0 0 0; H 0 0 1.4', basis = "cc-pvdz")

n_ao = mol.nao       # number of atomic orbitals 
n_el = mol.nelectron # number of electrons
n_occ = int(n_el/2)  # number of occupied orbitals 
e_nuc = mol.energy_nuc() # the nuclear repulsion energy

# overlap matrix
s = mol.intor('int1e_ovlp_sph') # get overap matrix

# show the overlap matrix
plt.figure(figsize = (6,6))
plt.imshow(s)
plt.show()

In [ ]:
# overlap matrix using PySCF
s = mol.intor('int1e_ovlp_sph') # get overap matrix

# kinetic and nuclear terms using PySCF
t = mol.intor_symmetric('int1e_kin') # kinetic integrals
v = mol.intor_symmetric('int1e_nuc') # nuclear

# core Hamiltonian
h = t + v

# electron repulsion integrals using PySCF
eri = mol.intor('int2e_sph', aosym=1)

# Löwdin orthogonalization matrix
s_12 = lowdin_s12_matrix(s)

# compute density matrix
d = density(s_12.dot(np.eye(n_ao)), 5)

# construct fock matrix
f = build_fock_matrix(h, eri, d)

# compute energy
energy = np.einsum("kl, kl", d, h+f)

# convergence parameters
convergence_thresh = 1e-7

# Minimize the energy
for i in range(100):
    d, c = advance(h, eri, d, n_occ, s_12)
    energy_new = np.einsum("kl, kl", d, h+build_fock_matrix(h,eri,d))
    print("Iteration = ",i, "energy = ", energy + e_nuc, " Hartrees")
    if np.abs(energy_new - energy)<convergence_thresh:
        print("    Break: converged in", i, "iterations.")
        break
    energy = energy_new

In [ ]:
# extract the basis as braketlab kets
basis = extract_pyscf_basis(mol)

In [ ]:
# form a linear combination of atomic orbitals as a braketlab ket

p = 2

phi0 = c[0,p]*basis[0]

for alpha in range(1, c.shape[0]):
    phi0 += c[alpha, p]*basis[alpha]
    
# view results interactively  
m = phi0.view(n_concentric = 200) #squared = False)
m